# 02 · Baseline and review

**Question:** Does adding rule/example context improve the lexical reference?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Controlled feature comparison
`comment_only` uses comment TF-IDF with logistic regression. `rule_examples` adds comment-to-rule/example cosine similarities, positive/negative maximum similarity, and their margin. Both are evaluated under the recorded split design. This is a lexical reference, not a claim of deep rule understanding.

In [2]:
display(metric_table(baseline["results"]))

,Model,Validation,Rule macro AUC,Log loss,Brier,Average precision
0,Comment-only TF-IDF,Familiar rules,0.7281,0.6148,0.2131,0.7170
1,Rule/example TF-IDF,Familiar rules,0.7287,0.6162,0.2139,0.7202
2,Comment-only TF-IDF,Held-out rule,0.6041,0.6731,0.2400,0.6108
3,Rule/example TF-IDF,Held-out rule,0.6156,0.6736,0.2405,0.6241


![Lexical validation comparison](../reports/baseline/comparison.svg)

## Held-out-rule feature effect
The difference below is descriptive. It is not a significance claim or a score from Kaggle.

In [3]:
held = {r["model"]: r["metrics"] for r in baseline["results"] if r["protocol"] == "heldout_rule"}
display(pd.DataFrame([{"Metric": metric, "Rule/example minus comment-only": held["rule_examples"][metric] - held["comment_only"][metric]}
    for metric in ["rule_macro_auc", "log_loss", "brier"]]).round(4))

,Metric,Rule/example minus comment-only
0,rule_macro_auc,0.0115
1,log_loss,0.0004
2,brier,0.0005


## Interpretation and reproducibility
Rule/example features modestly improve held-out ranking, while probability losses do not improve. The lexical model remains the reference for later candidates. Coefficients describe association, not causal effects. Raw examples and row-level errors are intentionally absent from the public notebook.

Training is explicit: `uv run jigsaw baseline --cloud`. That command fits or resumes a source-fingerprinted experiment; it is not required to read these results. Valid completed folds are reused, but an interrupted CPU solver restarts its active fold. Source changes can create a new experiment identity.

[03 · Results](03_saved_results.ipynb) is the consolidated employer overview.